# <font color="brown">Loan Default Model Comparison: Manual Loop vs. PyCaret </font>

## <font color = "brown">Problem Statement </font>

### <font color="blue"> Context

We already have the loan default dataset from the Feature Engineering case study, 6,000 loan applications, and we already know logistic regression works reasonably well on it. But every case study so far has tested exactly one algorithm at a time. A natural question for any real project: with six or more algorithms worth trying (the ones covered across this whole case-study series), how do we compare them all fairly, without writing the same evaluation code six times over? And once we've written that loop by hand, what does a dedicated AutoML tool like PyCaret actually add on top of it?

### <font color="blue"> Objective

- Build a manual model-comparison loop, by hand, using plain scikit-learn, across six familiar algorithms.
- Run the same comparison through PyCaret's low-code API and check, honestly, whether it agrees with the manual result.
- Identify specifically what PyCaret automates beyond model comparison, and what it costs to get that convenience.

### <font color="blue"> Data Dictionary

Same loan application data as the Feature Engineering case study (Age, Annual_Income, Loan_Amount, Credit_Score, Employment_Years, Num_Dependents, Existing_Loans, Home_Ownership, Purpose, and the target Default), reused here via a relative path, with **no engineered features added**, this case study is about comparing algorithms, not features.

### <font color="blue"> A Note on the Environment

PyCaret pins fairly old versions of numpy, pandas, and matplotlib, versions older than every other notebook in this series relies on. Rather than downgrading those packages globally (which would risk breaking every previously verified notebook), **this notebook runs in its own isolated virtual environment** (`PyCaret/.venv`), registered as the Jupyter kernel **"Python (PyCaret venv)"**. Select that kernel before running this notebook, every other notebook in this repository keeps using the regular shared environment untouched.

## <font color="brown"> Importing Necessary Libraries

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, precision_score, f1_score

%matplotlib inline

## <font color="brown"> Reading the Data

In [ ]:
df = pd.read_csv(r"../Feature Engineering/loan_applications.csv").drop(columns=["Applicant_ID"])
df.shape

In [ ]:
df.head()

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=1, stratify=df["Default"]
)
print("Train:", train_df.shape, " Test:", test_df.shape)

This exact split (`random_state=1`) is reused for **both** the manual loop and PyCaret below, so the two approaches are compared on identical data, a fair, apples-to-apples test.

## <font color="brown"> Part 1: A Manual Model-Comparison Loop

Six algorithms from across this case-study series, compared the same way we've compared models throughout: 10-fold cross-validation on the training set, tracking accuracy, AUC, recall, precision, and F1 for each.

In [ ]:
cat_cols = ["Home_Ownership", "Purpose"]
X_train = pd.get_dummies(train_df.drop(columns=["Default"]), columns=cat_cols, drop_first=True, dtype=int)
X_test = pd.get_dummies(test_df.drop(columns=["Default"]), columns=cat_cols, drop_first=True, dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df["Default"]
y_test = test_df["Default"]

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=1),
    "Decision Tree": DecisionTreeClassifier(random_state=1, max_depth=6),
    "Random Forest": RandomForestClassifier(random_state=1, n_estimators=200),
    "Gradient Boosting": GradientBoostingClassifier(random_state=1),
    "SVM (RBF)": SVC(probability=True, random_state=1),
    "Naive Bayes": GaussianNB(),
}

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)
scoring = {"Accuracy": "accuracy", "AUC": "roc_auc", "Recall": "recall", "Precision": "precision", "F1": "f1"}

rows = []
for name, model in models.items():
    res = cross_validate(model, X_train_s, y_train, cv=cv, scoring=scoring)
    rows.append({
        "Model": name,
        "Accuracy": res["test_Accuracy"].mean(),
        "AUC": res["test_AUC"].mean(),
        "Recall": res["test_Recall"].mean(),
        "Precision": res["test_Precision"].mean(),
        "F1": res["test_F1"].mean(),
    })

manual_leaderboard = pd.DataFrame(rows).sort_values("AUC", ascending=False).reset_index(drop=True)
manual_leaderboard.round(4)

**Gradient Boosting comes out on top by AUC (0.902), with Logistic Regression a close second (0.895).** This entire loop, six algorithms, five metrics each, cross-validated, took about 20 lines of code, and every one of those lines had to be written and reasoned about by hand: the encoding, the scaling, the CV strategy, the scoring dictionary, the sorting.

In [ ]:
gb_manual = GradientBoostingClassifier(random_state=1)
gb_manual.fit(X_train_s, y_train)
manual_test_pred = gb_manual.predict(X_test_s)
manual_test_proba = gb_manual.predict_proba(X_test_s)[:, 1]

manual_test_metrics = {
    "Accuracy": accuracy_score(y_test, manual_test_pred),
    "AUC": roc_auc_score(y_test, manual_test_proba),
    "Recall": recall_score(y_test, manual_test_pred),
    "Precision": precision_score(y_test, manual_test_pred),
    "F1": f1_score(y_test, manual_test_pred),
}
manual_test_metrics

## <font color="brown"> Part 2: The Same Comparison, via PyCaret

PyCaret's `setup()` handles missing values, encoding, and scaling automatically from a single call, and `compare_models()` cross-validates every model in its library, not just the six we picked by hand:

In [ ]:
from pycaret.classification import setup, compare_models, pull, predict_model, finalize_model, tune_model

In [ ]:
s = setup(data=train_df, target="Default", session_id=1, verbose=False, html=False)
best_model = compare_models(sort="AUC", verbose=False)
pycaret_leaderboard = pull()
pycaret_leaderboard[["Model", "Accuracy", "AUC", "Recall", "Prec.", "F1"]]

**PyCaret's own top pick is also Gradient Boosting (AUC 0.898), matching the manual loop's winner exactly**, out of 13 algorithms it tried automatically (versus the 6 we picked by hand), including several we never tested ourselves: Ridge Classifier, Linear Discriminant Analysis, AdaBoost, Extra Trees, all of which also scored competitively. The bottom of the table includes a Dummy Classifier, PyCaret's own built-in version of the naive baseline from the Imbalanced Dataset case study, included automatically as a sanity floor every other model should beat.

### <font color="blue"> Confirming the Winner on the Same Held-Out Test Set

In [ ]:
final_model = finalize_model(best_model)
test_preds = predict_model(final_model, data=test_df)
pycaret_test_metrics = pull()
pycaret_test_metrics[["Model", "Accuracy", "AUC", "Recall", "Prec.", "F1"]]

Compare this directly to the manual loop's Gradient Boosting result on the exact same test set:

In [ ]:
comparison = pd.DataFrame({
    "Manual Loop (Gradient Boosting)": manual_test_metrics,
    "PyCaret (Gradient Boosting)": {
        "Accuracy": pycaret_test_metrics["Accuracy"].iloc[0],
        "AUC": pycaret_test_metrics["AUC"].iloc[0],
        "Recall": pycaret_test_metrics["Recall"].iloc[0],
        "Precision": pycaret_test_metrics["Prec."].iloc[0],
        "F1": pycaret_test_metrics["F1"].iloc[0],
    },
}).round(4)
comparison

**Nearly identical**, recall matches to four decimal places (0.6838 both), and every other metric is within a rounding error. PyCaret didn't discover a better model than the manual loop, it found the same one, through its own independent preprocessing and cross-validation pipeline, in a fraction of the code. That agreement is itself the useful result: it's a second, independently-built confirmation that Gradient Boosting really is the right choice here, not an artifact of how our manual loop happened to preprocess the data.

## <font color="brown"> Part 3: What Else Does PyCaret Automate?

### <font color="blue"> One-Line Hyperparameter Tuning

In [ ]:
tuned_model = tune_model(best_model, optimize="AUC", verbose=False)
tuned_cv_metrics = pull()
tuned_cv_metrics.tail(3)

Worth reporting honestly: automatic tuning's mean cross-validated AUC (0.893) came in slightly *below* the untuned model's own CV score from Part 2 (0.898). Automated tuning is a randomized search over a fixed budget, it isn't guaranteed to beat sensible defaults, especially when the untuned model was already close to its ceiling. This is a genuinely useful, honest finding: `tune_model()` is worth trying, but its result should always be checked against the untuned baseline, exactly as we just did, rather than trusted automatically.

### <font color="blue"> One-Line Diagnostic Plots

In [ ]:
from pycaret.classification import plot_model

In [ ]:
plot_model(best_model, plot="auc")

In [ ]:
plot_model(best_model, plot="feature")

The ROC curve and feature-importance chart above would each normally take several lines of matplotlib code (as in nearly every earlier case study in this series), here, one line each.

### <font color="blue"> A Deployable Pipeline in One Call

In [ ]:
from pycaret.classification import save_model
save_model(final_model, "loan_default_gb_pipeline")

This saves the *entire* pipeline, encoding, scaling, and the fitted model together, as a single file. A new, raw loan application can be scored directly, without manually re-implementing the encoding and scaling steps used during training, a common, easy-to-get-wrong source of bugs when done by hand.

## <font color="brown"> Business Insights and Recommendations

- **PyCaret and the manual loop agreed on the winning model, and nearly exactly on that model's held-out performance.** This is strong, independently-verified evidence that Gradient Boosting is genuinely the right choice for this data, not an artifact of one particular preprocessing approach.

- **PyCaret's real value is breadth and speed, not a fundamentally different answer.** It tried 13 algorithms to our 6, including several (Ridge, LDA, AdaBoost) worth knowing about that we hadn't covered, and did it, encoding through leaderboard, in a fraction of the code the manual loop needed.

- **Automated tuning is not automatically an improvement, always check it against the untuned baseline.** Our tuned model's cross-validated AUC was actually slightly lower than the untuned default, a reminder that low-code tools still need the same skepticism as anything else.

- **The convenience has a real cost, worth planning for**: PyCaret's dependency requirements (older numpy, pandas, matplotlib) conflicted directly with every other package already installed for this project, we had to build an entirely separate virtual environment just to run this one notebook. Any team adopting PyCaret should plan for this kind of environment isolation up front, not discover it mid-project.

- **Recommended workflow**: use a tool like PyCaret early, for fast, broad model screening across many algorithms at once, then drop back to plain scikit-learn (as in every other notebook in this series) once a short list of promising models is identified, for full control over preprocessing, tuning, and interpretation of the final, production-bound model.